In [9]:
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download("uciml/glass")
file_path = os.path.join(path, "glass.csv")

df = pd.read_csv(file_path)
df.head()

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1


# 1. Pré-processamento e Limpeza dos Dados

Nesta etapa, serão realizadas verificações e tratamentos nos dados, incluindo:
- Remoção de duplicatas
- Análise de valores ausentes
- Tratamento de outliers
- Padronização dos dados

In [40]:
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download("uciml/glass")
file_path = os.path.join(path, "glass.csv")

df = pd.read_csv(file_path)

df.head()

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1


In [41]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 214 entries, 0 to 213
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   RI      214 non-null    float64
 1   Na      214 non-null    float64
 2   Mg      214 non-null    float64
 3   Al      214 non-null    float64
 4   Si      214 non-null    float64
 5   K       214 non-null    float64
 6   Ca      214 non-null    float64
 7   Ba      214 non-null    float64
 8   Fe      214 non-null    float64
 9   Type    214 non-null    int64  
dtypes: float64(9), int64(1)
memory usage: 16.8 KB


## Verificação se há duplicadas
Como ocorre: A função duplicated verifica se há duplicação, caso seja True equivale a 1 e caso seja False equivale a 0. E o SUM() é uma função no python que soma tudo

In [42]:
duplicados = df.duplicated().sum()
print("Número de duplicatas:", duplicados)



Número de duplicatas: 1


O resultado mostra que tem duas linhas iguais

In [43]:
df = df.drop_duplicates() # remorção da duplicata



In [44]:
duplicados = df.duplicated().sum()
print(duplicados)



0


## Remorção de colunas com alta taxa de valores ausentes (threshold >
50%).


verifica se a coluna tá vazia, calcula proporção de True/(True+False) e multiplica vezes 100 para transformar em porcentagem

In [45]:
missing_percent = df.isnull().mean() * 100 
print(missing_percent)

RI      0.0
Na      0.0
Mg      0.0
Al      0.0
Si      0.0
K       0.0
Ca      0.0
Ba      0.0
Fe      0.0
Type    0.0
dtype: float64


ou seja: Em todas as colunas não há nenhum valor ausente

## Imputação de Valores Faltantes

A etapa de imputação de valores faltantes não foi necessária, uma vez que o dataset não apresenta valores ausentes em nenhuma de suas colunas.

Caso houvesse valores faltantes, poderiam ser utilizadas estratégias como média, mediana ou moda, dependendo da distribuição dos dados.

## Tratamento de Outliers

Foi utilizado o método Z-score para identificação de outliers, seguindo a abordagem apresentada no material da disciplina.

O Z-score mede quantos desvios padrão um valor está distante da média da variável. Neste trabalho, foram considerados possíveis outliers os valores com Z-score absoluto maior que 2.

A identificação de outliers é importante porque valores muito distantes da distribuição principal podem influenciar o treinamento da MLP, dificultando a convergência e afetando os ajustes dos pesos da rede neural.

In [ ]:
features = df.drop("Type", axis=1)  #é qual a direção q eu quero mexer 0=linha 1=coluna

z_scores = (features - features.mean()) / features.std() #(valor - média) / desvio padrão

outliers_z = (z_scores.abs() > 2).sum() #abs() é o módulo no número e o sum soma todos os True

print(outliers_z)

RI    12
Na     9
Mg     0
Al    12
Si    12
K      3
Ca    12
Ba    15
Fe    12
dtype: int64


## Tratamento de Outliers

Foi utilizado o método Z-score para identificação de outliers, considerando valores com Z-score absoluto maior que 2.

Foram identificados outliers em diversas variáveis do dataset, como RI, Na, Al, Si, Ca, Ba e Fe, enquanto algumas variáveis, como Mg, não apresentaram valores discrepantes.

Apesar da presença de outliers, optou-se por não removê-los, uma vez que o dataset possui um número reduzido de amostras. A remoção desses valores poderia resultar em perda de informação relevante para o modelo.

Além disso, os outliers podem influenciar o treinamento da rede neural MLP, dificultando a convergência devido à influência desproporcional nos ajustes de peso.

## Normalização / Padronização dos Dados

Foi utilizada a técnica de padronização com StandardScaler, baseada no método Z-score, conforme apresentado no material da disciplina.

Essa abordagem transforma os dados para que apresentem média igual a 0 e desvio padrão igual a 1.

A padronização é essencial para redes neurais, como a MLP, pois essas são sensíveis à escala das variáveis de entrada. Quando as features possuem escalas diferentes, variáveis com valores maiores podem dominar o processo de aprendizado, prejudicando o desempenho do modelo.

Com a padronização, todas as variáveis passam a contribuir de forma equilibrada para o treinamento, favorecendo uma melhor convergência e estabilidade do modelo.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ============================
# 1. Separar variáveis
# ============================

# X = variáveis de entrada (features)
X = df.drop("Type", axis=1)

# y = variável alvo (target)
y = df["Type"]


# 2. Dividir treino e teste

# 80% treino, 20% teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# 3. Aplicar padronização


# Cria o scaler (baseado em Z-score)
scaler = StandardScaler()

# Aprende a escala com os dados de treino e transforma
X_train = scaler.fit_transform(X_train)

# Aplica a mesma transformação nos dados de teste
X_test = scaler.transform(X_test)

# ============================
# Fim da normalização
# ============================